# Generate Meta-path2vec Embeddings for PrimeKG（PyG 原生实现）

使用 **PyTorch Geometric 内置的 `MetaPath2Vec`**，基于 PrimeKG (`kg.csv`) 为所有节点生成 128 维嵌入。
输出格式与原 `node2vec_embeddings.pkl` 完全一致，可直接替换使用。

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict

import torch
from torch_geometric.nn import MetaPath2Vec
from tqdm import tqdm

## 2. 配置参数

In [2]:
KG_FILE    = '../data/kg.csv'
OUTPUT_PKL = '../data/metapath2vec_embeddings.pkl'

EMBED_DIM   = 128   # 与原代码 NODE2VEC_DIM=128 一致
WALK_LENGTH = 50    # 每条游走序列长度
CONTEXT_SIZE = 7    # Skip-gram 上下文窗口
WALKS_PER_NODE = 5  # 每节点游走次数
BATCH_SIZE  = 128
EPOCHS      = 5
LR          = 0.01

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

使用设备: cuda:0


## 3. 加载 kg.csv，构建 PyG 异构图所需的边索引

In [3]:


print('加载 kg.csv ...')
df = pd.read_csv(KG_FILE, low_memory=False)
print(f'总边数: {len(df):,}')

# ── 1. 预处理：构建原始 ID 字符串列 ──────────────────────
# 利用向量化字符串操作，瞬间完成，无需循环
df['src_raw'] = df['x_type'] + '::' + df['x_index'].astype(str)
df['dst_raw'] = df['y_type'] + '::' + df['y_index'].astype(str)

# ── 2. 构建全局映射字典 (用于最终输出) ──────────────────────
# 这一步是为了保留原代码的逻辑，方便后续保存结果
# 我们使用 Pandas 的 factorize 来快速生成整数 ID
node_local_id = defaultdict(dict)

# 获取所有唯一的节点类型
node_types = set(df['x_type'].unique()) | set(df['y_type'].unique())

for ntype in node_types:
    # 筛选出当前类型的节点
    mask_src = df['x_type'] == ntype
    mask_dst = df['y_type'] == ntype
    mask = mask_src | mask_dst
    
    # 提取该类型下的所有原始 ID
    raw_ids = pd.concat([df.loc[mask_src, 'src_raw'], df.loc[mask_dst, 'dst_raw']]).unique()
    
    # 建立映射：原始ID -> 整数ID (0, 1, 2...)
    # 这里的排序是为了保证结果的可复现性（可选）
    sorted_ids = sorted(raw_ids)
    node_local_id[ntype] = {raw_id: i for i, raw_id in enumerate(sorted_ids)}

# ── 3. 构建边索引 (向量化分组) ──────────────────────
edge_index_dict = {}

# 按边类型分组，这是 Pandas 最快的操作之一
# 我们只需要处理正向边，反向边稍后自动生成
grouped = df.groupby(['x_type', 'relation', 'y_type'])

print("正在构建边索引张量...")
for (src_type, rel, dst_type), group_df in grouped:
    # --- 正向边 ---
    # 使用 map 函数将字符串列快速转换为整数列
    src_indices = group_df['src_raw'].map(node_local_id[src_type]).values
    dst_indices = group_df['dst_raw'].map(node_local_id[dst_type]).values
    
    # 转为 Tensor
    edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)
    edge_index_dict[(src_type, rel, dst_type)] = edge_index

    # --- 反向边 (rev_) ---
    # 直接交换 src 和 dst 数组即可，无需重新查表
    rev_edge_index = torch.tensor([dst_indices, src_indices], dtype=torch.long)
    edge_index_dict[(dst_type, f'rev_{rel}', src_type)] = rev_edge_index

print(f'\n节点类型及数量:')
for ntype, mapping in sorted(node_local_id.items()):
    print(f'  {ntype}: {len(mapping):,}')
print(f'\n边类型数量: {len(edge_index_dict)}')

加载 kg.csv ...
总边数: 8,100,498
正在构建边索引张量...


/tmp/ipykernel_9602/3302700972.py:47: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)



节点类型及数量:
  anatomy: 14,035
  biological_process: 28,642
  cellular_component: 4,176
  disease: 17,080
  drug: 7,957
  effect/phenotype: 15,311
  exposure: 818
  gene/protein: 27,671
  molecular_function: 11,169
  pathway: 2,516

边类型数量: 100


## 4. 定义 Meta-path

PyG `MetaPath2Vec` 接受的 `metapath` 格式是**边类型元组的列表**：
```python
[(src_type, rel, dst_type), (dst_type, rel2, next_type), ...]
```
游走时会严格按此序列循环前进。

In [4]:
# 先查看实际存在的边类型，确认 meta-path 中用到的关系名称
print('实际边类型（前20条）:')
for etype in list(edge_index_dict.keys())[:20]:
    print(f'  {etype}')

实际边类型（前20条）:
  ('anatomy', 'anatomy_anatomy', 'anatomy')
  ('anatomy', 'rev_anatomy_anatomy', 'anatomy')
  ('anatomy', 'anatomy_protein_absent', 'gene/protein')
  ('gene/protein', 'rev_anatomy_protein_absent', 'anatomy')
  ('anatomy', 'anatomy_protein_present', 'gene/protein')
  ('gene/protein', 'rev_anatomy_protein_present', 'anatomy')
  ('biological_process', 'bioprocess_bioprocess', 'biological_process')
  ('biological_process', 'rev_bioprocess_bioprocess', 'biological_process')
  ('biological_process', 'bioprocess_protein', 'gene/protein')
  ('gene/protein', 'rev_bioprocess_protein', 'biological_process')
  ('biological_process', 'exposure_bioprocess', 'exposure')
  ('exposure', 'rev_exposure_bioprocess', 'biological_process')
  ('cellular_component', 'cellcomp_cellcomp', 'cellular_component')
  ('cellular_component', 'rev_cellcomp_cellcomp', 'cellular_component')
  ('cellular_component', 'cellcomp_protein', 'gene/protein')
  ('gene/protein', 'rev_cellcomp_protein', 'cellular_compo

In [5]:
# 查找连接特定节点类型对的所有关系，方便构造 meta-path
def find_relations(src_type, dst_type):
    return [
        etype for etype in edge_index_dict
        if etype[0] == src_type and etype[2] == dst_type
    ]

print('drug → disease 的关系:')
print(find_relations('drug', 'disease'))

print('\ndrug → gene/protein 的关系:')
print(find_relations('drug', 'gene/protein'))

print('\ngene/protein → gene/protein 的关系:')
print(find_relations('gene/protein', 'gene/protein'))

print('\ndrug → pathway 的关系:')
print(find_relations('drug', 'pathway'))

drug → disease 的关系:
[('drug', 'rev_contraindication', 'disease'), ('drug', 'rev_indication', 'disease'), ('drug', 'rev_off-label use', 'disease'), ('drug', 'contraindication', 'disease'), ('drug', 'indication', 'disease'), ('drug', 'off-label use', 'disease')]

drug → gene/protein 的关系:
[('drug', 'drug_protein', 'gene/protein'), ('drug', 'rev_drug_protein', 'gene/protein')]

gene/protein → gene/protein 的关系:
[('gene/protein', 'protein_protein', 'gene/protein'), ('gene/protein', 'rev_protein_protein', 'gene/protein')]

drug → pathway 的关系:
[]


In [6]:
# ── 根据上面输出的实际关系名称填写 meta-path ──────────────────
# 以下为基于 PrimeKG 常见关系名称的预设，如实际名称不同请根据上方输出修改

def first_rel(src, dst):
    """取两种节点类型之间的第一条正向边类型"""
    rels = find_relations(src, dst)
    # 优先取非 rev_ 开头的
    forward = [r for r in rels if not r[1].startswith('rev_')]
    return forward[0] if forward else rels[0]

# Drug-Disease 核心路径：Drug → Disease → Drug
drug_dis  = first_rel('drug', 'disease')
dis_drug  = first_rel('disease', 'drug')

# Drug-Gene 路径：Drug → Gene → Drug
drug_gene = first_rel('drug', 'gene/protein')
gene_drug = first_rel('gene/protein', 'drug')

# Disease-Gene 路径：Disease → Gene → Disease
dis_gene  = first_rel('disease', 'gene/protein')
gene_dis  = first_rel('gene/protein', 'disease')

# Gene-Gene 路径：Gene → Gene → Gene
gene_gene_rels = find_relations('gene/protein', 'gene/protein')
gene_gene = [r for r in gene_gene_rels if not r[1].startswith('rev_')]
gene_gene = gene_gene[0] if gene_gene else gene_gene_rels[0]

# Drug-Pathway 路径：Drug → Pathway → Drug
drug_path_rels = find_relations('drug', 'pathway')
path_drug_rels = find_relations('pathway', 'drug')

META_PATHS = []

# 1. Drug → Disease → Drug（最重要：直接对应 indication）
META_PATHS.append([drug_dis, dis_drug])

# 2. Disease → Drug → Disease
META_PATHS.append([dis_drug, drug_dis])

# 3. Drug → Gene → Drug
if drug_gene and gene_drug:
    META_PATHS.append([drug_gene, gene_drug])

# 4. Disease → Gene → Disease
if dis_gene and gene_dis:
    META_PATHS.append([dis_gene, gene_dis])

# 5. Gene → Gene → Gene（蛋白互作）
META_PATHS.append([gene_gene, gene_gene])

# 6. Drug → Pathway → Drug
if drug_path_rels and path_drug_rels:
    META_PATHS.append([drug_path_rels[0], path_drug_rels[0]])

# 7. Drug → Disease → Gene → Disease → Drug（长路径，捕获三元语义）
if drug_dis and dis_gene and gene_dis and dis_drug:
    META_PATHS.append([drug_dis, dis_gene, gene_dis, dis_drug])

print(f'构造了 {len(META_PATHS)} 条 meta-path：')
for mp in META_PATHS:
    path_str = ' → '.join(f"{e[0]}--[{e[1]}]-->{e[2]}" for e in mp)
    print(f'  {path_str}')

构造了 6 条 meta-path：
  drug--[contraindication]-->disease → disease--[contraindication]-->drug
  disease--[contraindication]-->drug → drug--[contraindication]-->disease
  drug--[drug_protein]-->gene/protein → gene/protein--[drug_protein]-->drug
  disease--[disease_protein]-->gene/protein → gene/protein--[disease_protein]-->disease
  gene/protein--[protein_protein]-->gene/protein → gene/protein--[protein_protein]-->gene/protein
  drug--[contraindication]-->disease → disease--[disease_protein]-->gene/protein → gene/protein--[disease_protein]-->disease → disease--[contraindication]-->drug


## 5. 为每条 Meta-path 训练 MetaPath2Vec

In [7]:
# 收集所有节点（带局部 ID）的完整 node_id 字符串，用于最终输出
# local_to_global[ntype][local_id] = "type::index" 字符串
local_to_global = {
    ntype: {v: k for k, v in mapping.items()}
    for ntype, mapping in node_local_id.items()
}

# 累积每个节点的嵌入（多条 meta-path 的平均）
# embed_accum["type::index"] = [emb1, emb2, ...]
embed_accum = defaultdict(list)

for mp_idx, metapath in enumerate(META_PATHS):
    src_type = metapath[0][0]  # 该 meta-path 的起始节点类型
    print(f'\n[{mp_idx+1}/{len(META_PATHS)}] 训练 meta-path: '
          f'{" → ".join(e[0] for e in metapath)} → {metapath[-1][2]}')

    mp2v = MetaPath2Vec(
        edge_index_dict=edge_index_dict,
        embedding_dim=EMBED_DIM,
        metapath=metapath,
        walk_length=WALK_LENGTH,
        context_size=CONTEXT_SIZE,
        walks_per_node=WALKS_PER_NODE,
        num_negative_samples=5,
        sparse=True,
    ).to(device)

    loader = mp2v.loader(
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
    )

    optimizer = torch.optim.SparseAdam(mp2v.parameters(), lr=LR)

    mp2v.train()
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0
        for pos_rw, neg_rw in tqdm(loader, desc=f'  Epoch {epoch}/{EPOCHS}', leave=False):
            optimizer.zero_grad()
            loss = mp2v.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'  Epoch {epoch}: loss = {total_loss / len(loader):.4f}')

    # 提取所有节点类型的嵌入并累积
    mp2v.eval()
    with torch.no_grad():
        for ntype, mapping in node_local_id.items():
            try:
                embs = mp2v(ntype)  # shape: (num_nodes_of_type, embed_dim)
                embs_np = embs.cpu().numpy().astype(np.float32)
                for local_id, emb in enumerate(embs_np):
                    global_id = local_to_global[ntype][local_id]
                    embed_accum[global_id].append(emb)
            except Exception:
                # 该 meta-path 未覆盖此节点类型，跳过
                pass

print('\n所有 meta-path 训练完成。')


[1/6] 训练 meta-path: drug → disease → drug


  Epoch 1: loss = 5.0091


  Epoch 2: loss = 2.9335


  Epoch 3: loss = 1.8559


  Epoch 4: loss = 1.3034


...

  Epoch 1: loss = 4.8382


  Epoch 2: loss = 3.3231


  Epoch 3: loss = 2.2640


  Epoch 4: loss = 1.6155


  Epoch 5: loss = 1.2301

所有 meta-path 训练完成。


## 6. 合并多条 Meta-path 的嵌入（取平均）

In [8]:
all_node_ids = [
    global_id
    for ntype, mapping in node_local_id.items()
    for global_id in mapping.keys()
]

records = []
missing = 0

for node_id in tqdm(all_node_ids, desc='合并嵌入'):
    emb_list = embed_accum.get(node_id, [])
    if emb_list:
        # 多条 meta-path 嵌入取平均
        emb = np.mean(emb_list, axis=0).astype(np.float32)
    else:
        # 未被任何 meta-path 覆盖：零向量（与原代码处理方式一致）
        emb = np.zeros(EMBED_DIM, dtype=np.float32)
        missing += 1
    records.append({'id': node_id, 'embedding': emb})

embed_df = pd.DataFrame(records)
print(f'总节点数:    {len(embed_df):,}')
print(f'有效嵌入:    {len(embed_df) - missing:,}')
print(f'零向量填充:  {missing:,}')

合并嵌入: 100%|██████████| 129375/129375 [00:01<00:00, 88709.72it/s] 

总节点数:    129,375
有效嵌入:    52,708
零向量填充:  76,667


## 7. 保存（与原 node2vec_embeddings.pkl 格式完全一致）

In [9]:
embed_df.to_pickle(OUTPUT_PKL)
print(f'已保存至: {OUTPUT_PKL}')

# 验证格式
verify_df   = pd.read_pickle(OUTPUT_PKL)
verify_dict = dict(zip(verify_df['id'], verify_df['embedding']))
sample_id   = list(verify_dict.keys())[0]
sample_emb  = verify_dict[sample_id]

print(f'\n格式验证:')
print(f'  id:       {sample_id}')
print(f'  shape:    {sample_emb.shape}')
print(f'  dtype:    {sample_emb.dtype}')
print(f'  均值/std: {sample_emb.mean():.4f} / {sample_emb.std():.4f}')
print('\n✓ 格式与原 node2vec_embeddings.pkl 兼容，可直接替换。')

已保存至: ../data/metapath2vec_embeddings.pkl

格式验证:
  id:       molecular_function::115051
  shape:    (128,)
  dtype:    float32
  均值/std: 0.0000 / 0.0000

✓ 格式与原 node2vec_embeddings.pkl 兼容，可直接替换。


## 附：在原代码中替换

只需修改 Cell 12 中的一行：

```python
# 原来
node2vec_df = pd.read_pickle('../data/node2vec_embeddings.pkl')

# 替换为
node2vec_df = pd.read_pickle('../data/metapath2vec_embeddings.pkl')
```